# FableForge Training Pipeline

This notebook walks through the complete training pipeline for FableForge models using **free resources** (Google Colab T4 GPU + Unsloth).

**Prerequisites:**
- Google Colab account (free tier works)
- Hugging Face account for dataset access
- ~8 hours of Colab GPU time

## 1. Install Dependencies

In [ ]:
# Install Unsloth and training dependencies
# Run this in Google Colab (T4 GPU runtime)
!pip install unsloth
!pip install trl transformers datasets accelerate peft bitsandbytes
!pip install huggingface_hub
!pip install trajectory-distiller

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Download Fable-5 Dataset

In [ ]:
from datasets import load_dataset

# Download the Fable-5 trajectory dataset
raw_dataset = load_dataset("fableforge/fable-5", split="train")

print(f"Total trajectories: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nSample trajectory:")
sample = raw_dataset[0]
print(f"  Task: {sample['task'][:100]}...")
print(f"  Category: {sample['category']}")
print(f"  Quality score: {sample.get('quality_score', 'N/A')}")
print(f"  Steps: {len(sample['trajectory'])}")

## 3. Convert Data with Trajectory Distiller

In [ ]:
from trajectory_distiller import TrajectoryConverter, QualityFilter

# Convert raw trajectories to ChatML training format
converter = TrajectoryConverter(
    format="chatml",
    include_thinking=True,     # Include reasoning chains
    include_tool_calls=True,    # Include tool invocations
    include_tool_results=True,  # Include tool outputs
    include_verification=True,  # Include verification results
)

train_data = converter.convert(raw_dataset)
train_data = train_data.train_test_split(test_size=0.05, seed=42)

print(f"Training samples: {len(train_data['train'])}")
print(f"Validation samples: {len(train_data['test'])}")

# Show a sample
sample = train_data['train'][0]
print(f"\nSample text (first 500 chars):")
print(sample['text'][:500])

In [ ]:
# Quality filter — keep only high-quality trajectories
quality_filter = QualityFilter(
    min_score=0.7,
    max_repetitions=3,
    require_verification=True,
    exclude_patterns=["infinite_loop", "repeated_failure", "no_tool_use"],
)

filtered_data = quality_filter.apply(train_data['train'])
print(f"Kept {len(filtered_data)} of {len(train_data['train'])} samples "
      f"({len(filtered_data)/len(train_data['train'])*100:.1f}%)")

## 4. Stage 1: Behavior Cloning

In [ ]:
from unsloth import FastLanguageModel

# Load base model with 4-bit quantization for T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-14B",
    max_seq_length=4096,
    load_in_4bit=True,      # QLoRA: essential for T4
    dtype=None,              # Auto-detect
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                   # LoRA rank
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print model info
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Model size: {total * 2 / 1e9:.1f}B (fp16)")
print(f"Effective LoRA size: {trainable * 2 / 1e6:.1f}M params")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Stage 1: Behavior Cloning training
stage1_args = TrainingArguments(
    output_dir="./fableforge-14b-stage1",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # Effective batch = 16
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    fp16=True,                          # T4: use fp16
    bf16=False,                          # T4: no bf16 support
    optim="adamw_8bit",
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=100,
    report_to="wandb",                 # Track with wandb
)

stage1_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=filtered_data,
    eval_dataset=train_data["test"],
    args=stage1_args,
    dataset_text_field="text",
    max_seq_length=4096,
)

print("Stage 1 training configuration:")
print(f"  Effective batch size: {2 * 8}")
print(f"  Learning rate: {2e-5}")
print(f"  Epochs: 3")
print(f"  Max sequence length: 4096")
print(f"  Estimated time on T4: ~2-4 hours")
print()
print("Uncomment the line below to start training:")
print("# stage1_trainer.train()")

## 5. Stage 2: Skill Fine-tuning

In [ ]:
from trajectory_distiller import SkillFilter

# Filter for skill-specific data
skill_filter = SkillFilter(
    categories=["tool_use", "verification", "code_generation", "debugging"],
    min_quality_score=0.8,
)

skill_data = skill_filter.filter(filtered_data)

# Generate synthetic verification data
from trajectory_distiller import create_verification_dataset

synthetic_verify = create_verification_dataset(
    base_data=skill_data,
    num_samples=5000,
    include_positive=True,
    include_negative=True,
)

from datasets import concatenate_datasets
combined_skill_data = concatenate_datasets([skill_data, synthetic_verify])

print(f"Skill training samples: {len(skill_data)}")
print(f"Synthetic verification samples: {len(synthetic_verify)}")
print(f"Combined: {len(combined_skill_data)}")

In [ ]:
# Load Stage 1 model and continue training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./fableforge-14b-stage1-adapter",
    max_seq_length=4096,
    load_in_4bit=True,
)

stage2_args = TrainingArguments(
    output_dir="./fableforge-14b-stage2",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,        # Lower LR for fine-tuning
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="adamw_8bit",
    save_steps=100,
    save_total_limit=3,
)

stage2_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined_skill_data,
    args=stage2_args,
    max_seq_length=4096,
)

print("Stage 2 configuration:")
print(f"  Learning rate: {1e-5} (lower than Stage 1)")
print(f"  Estimated time on T4: ~1-3 hours")
print("# stage2_trainer.train()")

## 6. Stage 3: Error Recovery Training

In [ ]:
from trajectory_distiller import ErrorRecoveryGenerator

# Generate error→fix pairs
recovery_gen = ErrorRecoveryGenerator(
    model_name="Qwen/Qwen2.5-14B",
    error_types=[
        "syntax_error",
        "runtime_error",
        "test_failure",
        "type_error",
        "import_error",
        "logic_error",
    ],
)

recovery_data = recovery_gen.generate(
    base_code=filtered_data,
    num_samples=3000,
)

print(f"Error recovery samples: {len(recovery_data)}")
print(f"\nSample error type distribution:")
for error_type, count in recovery_data.features['error_type'].value_counts().items():
    print(f"  {error_type}: {count}")

In [ ]:
# Stage 3 training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./fableforge-14b-stage2-adapter",
    max_seq_length=4096,
    load_in_4bit=True,
)

stage3_args = TrainingArguments(
    output_dir="./fableforge-14b-stage3",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,       # Even lower LR
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="adamw_8bit",
    save_steps=100,
    save_total_limit=3,
)

stage3_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=recovery_data,
    args=stage3_args,
    max_seq_length=4096,
)

print("Stage 3: Error Recovery Training")
print(f"  Learning rate: {5e-6}")
print(f"  Estimated time: ~1-2 hours on T4")
print("# stage3_trainer.train()")

## 7. Stage 4: DPO (Direct Preference Optimization)

In [ ]:
from trajectory_distiller import PreferencePairGenerator
from trl import DPOTrainer, DPOConfig

# Generate preference pairs
pair_gen = PreferencePairGenerator()
pairs = pair_gen.generate(
    tasks=filtered_data.select(range(1000)),
    num_variants=3,
    ranking_criteria=["correctness", "efficiency", "readability", "safety"],
)

print(f"Preference pairs: {len(pairs)}")

# DPO training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./fableforge-14b-stage3-adapter",
    max_seq_length=4096,
    load_in_4bit=True,
)

dpo_config = DPOConfig(
    output_dir="./fableforge-14b-stage4-dpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-7,       # Very low LR for DPO
    lr_scheduler_type="cosine",
    fp16=True,
    optim="adamw_8bit",
    beta=0.1,
    max_length=4096,
    max_prompt_length=2048,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=pairs,
    processing_class=tokenizer,
)

print("Stage 4: DPO Training")
print(f"  Beta: {0.1}")
print(f"  Learning rate: {5e-7}")
print(f"  Estimated time: ~2-4 hours on T4")
print("# dpo_trainer.train()")

## 8. Monitoring with wandb

In [ ]:
# Set up Weights & Biases for tracking
import wandb

# Login (use your API key or set WANDB_API_KEY env var)
# wandb.login(key="your-wandb-key")

# Or already set in TrainingArguments: report_to="wandb"
# Key metrics to watch:
print("Key wandb metrics to monitor:")
print("  train/loss          — Should decrease steadily")
print("  train/learning_rate — Should follow cosine schedule")
print("  eval/loss           — Should decrease, watch for overfitting")
print("  train/gpu_memory   — Should stay under 15GB on T4")
print("  train/samples_per_second — Training throughput")

# Example dashboard panels:
print("\n[bold]Recommended wandb panels:[/bold]")
print("  1. Loss curves (train + eval)")
print("  2. Learning rate schedule")
print("  3. GPU memory utilization")
print("  4. Per-stage comparison")
print("  5. Token throughput")

## 9. Export to Ollama

In [ ]:
# Merge LoRA adapter into base model
model.save_pretrained_merged(
    "./fableforge-14b-final",
    tokenizer=tokenizer,
    save_method="merged_16bit",
)

# Export as GGUF for Ollama
model.save_pretrained_gguf(
    "fableforge-14b",
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
)

print("Quantization options for GGUF export:")
print("  Q4_K_M  — ~5GB, good quality, fast (recommended)")
print("  Q5_K_M  — ~6GB, better quality, fast")
print("  Q8_0    — ~9GB, very good quality, medium speed")
print("  FP16    — ~14GB, best quality, slow")

# Create Ollama Modelfile
modelfile_content = """FROM ./fableforge-14b-Q4_K_M.gguf
PARAMETER temperature 0.7
PARAMETER num_ctx 8192
SYSTEM You are FableForge, an expert coding agent. You plan, execute, verify, and recover from errors when creating or modifying code. Always think step by step.
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("\nTo build and test in Ollama:")
print("  $ ollama create fableforge-14b -f Modelfile")
print("  $ ollama run fableforge-14b")

# Push to Hugging Face
print("\nTo push to Hugging Face:")
print("  model.push_to_hub('fableforge/fableforge-14b')")
print("  tokenizer.push_to_hub('fableforge/fableforge-14b')")

## 10. Summary

Complete training pipeline for FableForge-14B:

| Stage | Purpose | Time (T4) | LR |
|-------|---------|-----------|-----|
| 1. Behavior Cloning | Learn basic patterns | 2-4h | 2e-5 |
| 2. Skill Fine-tuning | Tool use & verification | 1-3h | 1e-5 |
| 3. Error Recovery | Fix failures | 1-2h | 5e-6 |
| 4. DPO | Output quality | 2-4h | 5e-7 |

**Total estimated time:** 6-13 hours on Colab T4 (free tier)

For ShellWhisperer-1.5B and ReasonCritic-7B, follow the same pipeline
with adjusted parameters (see docs/training-guide.md).